# Vanilla GAN (Generative Adversarial Network)

기본 GAN을 구현하고 MNIST 이미지를 생성합니다.

## 학습 목표
1. GAN의 원리 이해
2. Generator와 Discriminator 구현
3. 적대적 학습 프로세스
4. 이미지 생성

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 데이터 로드

In [ ]:
# MNIST 데이터셋
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1, 1] 범위로 정규화
])

dataset = torchvision.datasets.MNIST(
    root='../data',
    train=True,
    transform=transform,
    download=True
)

dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
print(f"데이터셋 크기: {len(dataset)}")

## 2. Generator 정의

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_shape=(1, 28, 28)):
        super(Generator, self).__init__()
        self.img_shape = img_shape
        
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            
            nn.Linear(1024, int(np.prod(img_shape))),
            nn.Tanh()  # [-1, 1] 출력
        )
    
    def forward(self, z):
        img = self.model(z)
        return img.view(img.size(0), *self.img_shape)

generator = Generator().to(device)
print(f"Generator 파라미터: {sum(p.numel() for p in generator.parameters()):,}")

## 3. Discriminator 정의

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_shape=(1, 28, 28)):
        super(Discriminator, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(int(np.prod(img_shape)), 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        return self.model(img_flat)

discriminator = Discriminator().to(device)
print(f"Discriminator 파라미터: {sum(p.numel() for p in discriminator.parameters()):,}")

## 4. 학습

In [ ]:
# 손실 함수 및 옵티마이저
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

latent_dim = 100
epochs = 50

# 학습 기록
g_losses, d_losses = [], []
fixed_noise = torch.randn(64, latent_dim).to(device)  # 시각화용

for epoch in range(epochs):
    g_loss_epoch, d_loss_epoch = 0, 0
    
    for real_imgs, _ in tqdm(dataloader, desc=f'Epoch {epoch+1}'):
        batch_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        
        # 레이블
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        # ---------------------
        # Discriminator 학습
        # ---------------------
        optimizer_D.zero_grad()
        
        # Real 이미지 손실
        real_loss = criterion(discriminator(real_imgs), real_labels)
        
        # Fake 이미지 생성 및 손실
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_imgs = generator(z)
        fake_loss = criterion(discriminator(fake_imgs.detach()), fake_labels)
        
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()
        
        # ---------------------
        # Generator 학습
        # ---------------------
        optimizer_G.zero_grad()
        
        # Generator가 Discriminator를 속이려 함
        g_loss = criterion(discriminator(fake_imgs), real_labels)
        g_loss.backward()
        optimizer_G.step()
        
        g_loss_epoch += g_loss.item()
        d_loss_epoch += d_loss.item()
    
    g_losses.append(g_loss_epoch / len(dataloader))
    d_losses.append(d_loss_epoch / len(dataloader))
    
    print(f'Epoch {epoch+1}/{epochs} | D Loss: {d_losses[-1]:.4f} | G Loss: {g_losses[-1]:.4f}')

In [ ]:
# 학습 곡선
plt.figure(figsize=(10, 5))
plt.plot(g_losses, label='Generator')
plt.plot(d_losses, label='Discriminator')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

## 5. 이미지 생성

In [ ]:
# 생성된 이미지 시각화
generator.eval()
with torch.no_grad():
    fake_imgs = generator(fixed_noise).cpu()

# 이미지를 [0, 1] 범위로 변환
fake_imgs = (fake_imgs + 1) / 2

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(fake_imgs[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Images')
plt.tight_layout()
plt.show()

In [ ]:
# 잠재 공간 보간
def interpolate(z1, z2, steps=10):
    ratios = np.linspace(0, 1, steps)
    vectors = []
    for ratio in ratios:
        v = z1 * (1 - ratio) + z2 * ratio
        vectors.append(v)
    return torch.stack(vectors)

z1 = torch.randn(1, latent_dim).to(device)
z2 = torch.randn(1, latent_dim).to(device)

interpolated = interpolate(z1, z2, steps=10)

with torch.no_grad():
    generated = generator(interpolated).cpu()
    generated = (generated + 1) / 2

plt.figure(figsize=(15, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(generated[i].squeeze(), cmap='gray')
    plt.axis('off')
plt.suptitle('Latent Space Interpolation')
plt.show()

## 연습 문제

1. DCGAN (Deep Convolutional GAN)을 구현해보세요.
2. Conditional GAN으로 특정 숫자를 생성해보세요.
3. CIFAR-10 데이터셋으로 컬러 이미지를 생성해보세요.
4. Mode collapse 문제를 해결하는 기법을 시도해보세요.